In [ ]:
import csv
import hashlib
import numpy as np
from gostcrypto import gosthash
from collect import collect
from nist_sts.run_tests import *

# Extract

In [ ]:
# collect(5000000, 'data.csv')

In [2]:
def load_data(filename):
    data = []
    with open(filename, 'r') as f:
        reader = csv.reader(f)
        next(reader, None)
        for row in reader:
            if row and row[1].isdigit():
                data.append(int(row[1]))

    print(f"Загружено {len(data)} отсчетов.")
    return data

In [3]:
def extract_compressed(raw_data, target_bits, hash_type="sha3_512", compression_ratio=2):
    diffs = np.diff(raw_data)
    raw_bits = (diffs & 1).astype(int)
    bits_list = raw_bits.tolist()

    bit_string = "".join(map(str, bits_list))
    
    input_block_size = 512 
    output_bits_per_hash = 512 // compression_ratio
    
    final_bits = []

    for i in range(0, len(bit_string) - input_block_size, input_block_size):
        if len(final_bits) >= target_bits:
            break
            
        block_str = bit_string[i : i + input_block_size]
        if not all(c in '01' for c in block_str):
            continue
            
        block_int = int(block_str, 2)
        block_bytes = block_int.to_bytes(input_block_size // 8, byteorder='big')

        if hash_type == "sha3_512":
            hash_obj = hashlib.sha3_512(block_bytes)
        elif hash_type == "blake2b":
            hash_obj = hashlib.blake2b(block_bytes, digest_size=64)
        elif hash_type == "streebog512":
            hash_obj = gosthash.new('streebog512', data=block_bytes)
        else:
            raise ValueError()

        hash_hex = hash_obj.hexdigest()
        hash_int = int(hash_hex, 16)
        hash_bin = format(hash_int, '0512b') 
        
        for b in hash_bin[:output_bits_per_hash]:
            if len(final_bits) >= target_bits:
                break
            final_bits.append(int(b))
            
    return np.array(final_bits, dtype=int)

In [8]:
data = load_data('data.csv')
extracted_bits = extract_compressed(data, 1000000, 'sha3_512', 3)
run_tests(extracted_bits)

Загружено 5000000 отсчетов.
Длина последовательности: 1000000 бит
Распределение: 0.5000 (1) / 0.5000 (0)
----------------------------------------------------------------------
Название теста                                | P-value    | Статус
----------------------------------------------------------------------
1. Frequency (Monobit) Test                   | 0.948970   | Пройден
2. Block Frequency Test                       | 0.187352   | Пройден
3. Runs Test                                  | 0.308676   | Пройден
4. Longest Run of Ones in a Block             | 0.421744   | Пройден
5. Binary Matrix Rank Test                    | 0.036662   | Пройден
6. Spectral (DFT) Test                        | 0.989387   | Пройден
7. Non-overlapping Template (mean P)          | 0.506664   | Пройден
8. Overlapping Template Matching (m=9)        | 0.278022   | Пройден
9. Maurer's Universal Test (L=6)              | 0.672381   | Пройден
10. Linear Complexity Test                    | 0.699264   | Про

In [9]:
data = load_data('data.csv')
extracted_bits = extract_compressed(data, 1000000, 'blake2b', 3)
run_tests(extracted_bits)

Загружено 5000000 отсчетов.
Длина последовательности: 1000000 бит
Распределение: 0.5000 (1) / 0.5000 (0)
----------------------------------------------------------------------
Название теста                                | P-value    | Статус
----------------------------------------------------------------------
1. Frequency (Monobit) Test                   | 0.979257   | Пройден
2. Block Frequency Test                       | 0.516723   | Пройден
3. Runs Test                                  | 0.744425   | Пройден
4. Longest Run of Ones in a Block             | 0.375872   | Пройден
5. Binary Matrix Rank Test                    | 0.882449   | Пройден
6. Spectral (DFT) Test                        | 0.192255   | Пройден
7. Non-overlapping Template (mean P)          | 0.521581   | Пройден
8. Overlapping Template Matching (m=9)        | 0.644874   | Пройден
9. Maurer's Universal Test (L=6)              | 0.898188   | Пройден
10. Linear Complexity Test                    | 0.289919   | Про

In [7]:
data = load_data('data.csv')
extracted_bits = extract_compressed(data, 1000000, 'streebog512', 3)
run_tests(extracted_bits)

Загружено 5000000 отсчетов.
Длина последовательности: 1000000 бит
Распределение: 0.5006 (1) / 0.4994 (0)
----------------------------------------------------------------------
Название теста                                | P-value    | Статус
----------------------------------------------------------------------
1. Frequency (Monobit) Test                   | 0.271332   | Пройден
2. Block Frequency Test                       | 0.841302   | Пройден
3. Runs Test                                  | 0.994247   | Пройден
4. Longest Run of Ones in a Block             | 0.352592   | Пройден
5. Binary Matrix Rank Test                    | 0.273924   | Пройден
6. Spectral (DFT) Test                        | 0.155205   | Пройден
7. Non-overlapping Template (mean P)          | 0.373578   | Пройден
8. Overlapping Template Matching (m=9)        | 0.090132   | Пройден
9. Maurer's Universal Test (L=6)              | 0.865270   | Пройден
10. Linear Complexity Test                    | 0.952728   | Про